*This notebook runs short numerical checks. The full study is reported in the [README](README.md), with saved results under [`results/`](results/) and the original study code at [`study-v1`](https://github.com/MoritzDue/bermudan-neural-stopping/tree/study-v1). The quick-mode estimates below are not part of the reported full study.*

# 01 — Method validation

This notebook checks the simulation and pricing methods before comparing LSM with neural stopping:

- Risk-neutral GBM simulation, discounting and exercise timing.
- A one-dimensional Bermudan put against binomial and convolution references.
- LSM behaviour across training sizes.
- A small neural-training and primal–dual calculation.

The checks provide numerical evidence of consistency, not a proof that every estimator is correct. Some larger validation experiments were omitted from the completed study.

In [ ]:
MODE = "quick"

import json, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from stopping import *

print("Thread limits:", limit_threads(1))
RUN_PAPER_CHECK = MODE == "quick"
from validation_sections import validation_checks, validation_put_fit, validation_convergence, validation_paper, validation_dual
BASE = CONFIG[MODE]["base_seed"]
OUT = results_dir(MODE)
print("Mode:", MODE, "| results ->", OUT)
cfg = CONFIG[MODE]["validation"]
CHECK_SEED = 500
cfg

## 1. Simulation and reference prices

Check GBM moments, correlation, seeded path generation and convergence of the binomial reference.

In [ ]:
globals().update(validation_checks())


## 2. One-dimensional Bermudan put

Compare independently evaluated LSM and neural policies with a refined CRR binomial value. In one dimension, sorting the asset input has no effect. The test-path interval measures Monte Carlo uncertainty for a fitted policy.

In [ ]:
globals().update(validation_put_fit(MODE, cfg, BASE, OUT, CHECK_SEED, put, put_ref))


## 3. LSM across training sizes

Fit LSM at several budgets, selecting its regression settings on separate validation paths. The training seeds share the same test paths, so their observed variation does not capture all test-set sampling uncertainty. This is a numerical consistency check rather than evidence of monotonic convergence.

In [ ]:
globals().update(validation_convergence(MODE, cfg, BASE, OUT, CHECK_SEED))


## 4. Neural stopping with freshly simulated training paths

This check trains with new simulated paths at each optimisation step, using the architecture and payoff input described by Becker et al. (2019). The short run exercises this implementation but is too small to support a pricing comparison with the published benchmark. The larger paper-regime check was not completed.

In [ ]:
if RUN_PAPER_CHECK:
    globals().update(validation_paper(cfg, BASE, OUT, CHECK_SEED))
else:
    print("Paper-regime implementation check not tested (full unit budget).")


## Interpretation

The one-dimensional references provide independent numerical comparisons. Discretisation error in the references and Monte Carlo error in estimated policy values are different sources of uncertainty. The completed five-dimensional comparisons are reported in notebook 03 and the README; the runs above should not be treated as additional full-study observations.

## 5. Primal–dual upper bound

For a fixed exercise policy, let `Z_k = exp(-r t_k) g(S_k)` be the discounted exercise payoff and let `C_k` denote its expected future payoff under that policy, conditional on the state at date `k`.

Define `L_k = Z_k` when the policy exercises, and `L_k = C_k` when it continues. Then `M_0 = 0` and

```text
M_(k+1) - M_k = L_(k+1) - C_k
```

are martingale increments under exact conditional expectations. For any stopping rule, the expected maximum of `Z_k - M_k` across exercise dates bounds the optimal stopping value from above.

The implementation estimates the conditional expectations using independent inner simulations. The finite-inner dual estimate preserves the upper-bound property **in expectation**, but may be biased upward. Neither an individual simulated value nor its confidence interval endpoint is guaranteed to bound the true price. See [`dual.py`](dual.py) and [`RESEARCH_NOTES.md`](RESEARCH_NOTES.md).

## 6. One-dimensional dual check

Evaluate fitted policies on independent outer paths and compare primal–dual estimates with the CRR reference. Varying the inner simulation count tests sensitivity to nested-simulation error, but does not establish that this bias is zero.

In [ ]:
globals().update(validation_dual(MODE, cfg, BASE, OUT, CHECK_SEED, put, put_ref))


## Saved validation results

The completed validation summary is in [`results/supporting/summary.json`](results/supporting/summary.json). For the Bermudan put, the refined CRR value is **6.02753210** and the independent convolution value is **6.02757520**. The reported numerical-grid band (**0.00005382**) is a heuristic, not a certified pricing-error bound. The larger neural paper-regime check and the proposed 2D validation were not completed.